In [1]:
import pandas as pd
import os

# Define the file path (update if your downloaded filename is different)
file_path = 'dataset/spam.csv'

# Check if the file exists before loading
if os.path.exists(file_path):
    print("Loading Spam SMS dataset...")
    # The SMS dataset often requires latin-1 encoding
    df = pd.read_csv(file_path, encoding='latin-1')
    
    # Display the first few rows and dataset information
    display(df.head())
    print("\nDataset Info:")
    df.info()
else:
    print(f"File not found at {file_path}. Please check your dataset folder.")

Loading Spam SMS dataset...


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN



Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 677.1 KB


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print("Cleaning the dataset...")
# Drop the empty unnamed columns
df_cleaned = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)

# Rename columns for better readability
df_cleaned = df_cleaned.rename(columns={'v1': 'label', 'v2': 'message'})

# Map text labels to binary numbers (ham = 0, spam = 1)
df_cleaned['label_num'] = df_cleaned['label'].map({'ham': 0, 'spam': 1})

# Define features (X) and target (y)
X = df_cleaned['message']
y = df_cleaned['label_num']

# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Converting text messages to numerical features using TF-IDF...")
# Initialize TF-IDF Vectorizer (removing standard English stop words like 'the', 'is', 'and')
tfidf = TfidfVectorizer(stop_words='english', max_features=3000)

# Fit on training data and transform both train and test sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Training features shape: {X_train_tfidf.shape}")
print(f"Testing features shape: {X_test_tfidf.shape}")
display(df_cleaned.head())

Cleaning the dataset...
Converting text messages to numerical features using TF-IDF...
Training features shape: (4457, 3000)
Testing features shape: (1115, 3000)


,label,message,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [3]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

print("Training Multinomial Naive Bayes model...")
# Initialize the model
model = MultinomialNB()

# Train the model
model.fit(X_train_tfidf, y_train)

print("Making predictions on the test set...")
# Predict on the unseen testing data
y_pred = model.predict(X_test_tfidf)

# Calculate and display the results
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)']))

Training Multinomial Naive Bayes model...
Making predictions on the test set...

Test Accuracy: 97.85%

Classification Report:
              precision    recall  f1-score   support

     Ham (0)       0.98      1.00      0.99       965
    Spam (1)       1.00      0.84      0.91       150

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [4]:
print("Saving predictions to CSV...")

# Create a DataFrame combining the original test messages with actual and predicted labels
test_results = pd.DataFrame({
    'Message': X_test, 
    'Actual_Label': y_test, 
    'Predicted_Label': y_pred
})

# Save to the dataset folder
output_path = 'dataset/predicted_spam.csv'
test_results.to_csv(output_path, index=False)

print(f"Task 4 Complete! Predictions successfully saved to {output_path}")
display(test_results.head())

Saving predictions to CSV...
Task 4 Complete! Predictions successfully saved to dataset/predicted_spam.csv


,Message,Actual_Label,Predicted_Label
3245,"Funny fact Nobody teaches volcanoes 2 erupt, t...",0,0
944,I sent my scores to sophas and i had to do sec...,0,0
1044,We know someone who you know that fancies you....,1,1
2484,Only if you promise your getting out as SOON a...,0,0
812,Congratulations ur awarded either å£500 of CD ...,1,1
